# Train and Save a Logistic Regression Model — Bank Marketing

This workbook creates the model artifacts used by `app_logistic_regression.py`. It follows the same practical workflow as the class notebook: data quality check, preprocessing, logistic regression model fitting, evaluation, coefficient interpretation, and artifact export.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

RANDOM_STATE = 42
APP_DIR = Path.cwd()
DATA_DIR = APP_DIR / "data"
ARTIFACT_DIR = APP_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

## 2. Helper functions

In [ ]:
def summarize_missing_values(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str)
    }).sort_values("missing_percent", ascending=False)

def predict_with_threshold(probabilities, threshold=0.5):
    return (probabilities >= threshold).astype(int)

def evaluate_binary_classifier(y_true, y_pred, y_proba, label="Model"):
    return pd.DataFrame([{
        "model": label,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }])

def plot_confusion_matrix(y_true, y_pred, title="Confusion matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 3. Load and clean the bank marketing dataset

The target `y` indicates whether the customer responded positively. Fully blank rows and index-like export columns are removed before modeling.

In [ ]:
bank_path = DATA_DIR / "bank_marketing_sample.csv"
if not bank_path.exists():
    raise FileNotFoundError("Expected data/bank_marketing_sample.csv")

bank = pd.read_csv(bank_path, skip_blank_lines=True)
print("Initial shape:", bank.shape)
bank.head()

In [ ]:
empty_rows_before = bank.isna().all(axis=1).sum()
bank = bank.dropna(how="all").copy()
index_like_cols = [col for col in bank.columns if col.lower().startswith("unnamed")]
bank_clean = bank.drop(columns=index_like_cols, errors="ignore")

print("Fully empty rows before cleaning:", empty_rows_before)
print("Dropped index-like columns:", index_like_cols)
print("Shape after cleaning:", bank_clean.shape)
bank_clean.head()

## 4. Data understanding and quality checks

Even if the dataset looks clean, the workflow explicitly checks missing values, duplicates and target balance.

In [ ]:
print("Duplicate rows:", bank_clean.duplicated().sum())
display(summarize_missing_values(bank_clean))
display(bank_clean.describe(include="all").T)

In [ ]:
target_balance = bank_clean["y"].value_counts(normalize=True).rename("share").to_frame()
target_balance["count"] = bank_clean["y"].value_counts()
target_balance

In [ ]:
plt.figure(figsize=(5, 4))
bank_clean["y"].value_counts().sort_index().plot(kind="bar")
plt.title("Bank marketing target distribution")
plt.xlabel("Positive response (1 = yes)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## 5. Prepare data and train logistic regression

Categorical variables are one-hot encoded. Numeric variables, if present in future versions of the data, are median-imputed and standardized. `class_weight="balanced"` is used because positive responses are the minority class.

In [ ]:
X = bank_clean.drop(columns="y")
y = bank_clean["y"].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

In [ ]:
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess = ColumnTransformer(transformers=[
    ("cat", categorical_preprocess, categorical_features),
    ("num", numeric_preprocess, numeric_features),
])

bank_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

bank_logreg.fit(X_train, y_train)

valid_proba = bank_logreg.predict_proba(X_valid)[:, 1]
valid_pred = predict_with_threshold(valid_proba, threshold=0.5)
evaluate_binary_classifier(y_valid, valid_pred, valid_proba, label="Bank marketing logistic regression")

## 6. Evaluate predictions and threshold behavior

In [ ]:
plot_confusion_matrix(y_valid, valid_pred, "Bank marketing: default threshold 0.5")
print(classification_report(y_valid, valid_pred, target_names=["no response", "positive response"], zero_division=0))

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = []
for threshold in thresholds:
    pred = predict_with_threshold(valid_proba, threshold)
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_valid, pred),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0),
        "flagged_share": pred.mean(),
    })
threshold_df = pd.DataFrame(threshold_results)
threshold_df.head()

In [ ]:
plt.figure(figsize=(9, 5))
for metric in ["precision", "recall", "f1", "flagged_share"]:
    plt.plot(threshold_df["threshold"], threshold_df[metric], marker="o", label=metric)
plt.title("Threshold tuning: business trade-off")
plt.xlabel("Decision threshold")
plt.ylabel("Score / share")
plt.legend()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_valid, valid_proba)
auc_value = roc_auc_score(y_valid, valid_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="No-skill baseline")
plt.title("ROC curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.legend()
plt.show()

## 7. Interpret coefficients

In [ ]:
encoded_feature_names = bank_logreg.named_steps["preprocess"].get_feature_names_out()
bank_coef = pd.DataFrame({
    "feature": encoded_feature_names,
    "coefficient": bank_logreg.named_steps["model"].coef_[0],
})
bank_coef["abs_coefficient"] = bank_coef["coefficient"].abs()
bank_coef = bank_coef.sort_values("abs_coefficient", ascending=False)
bank_coef.head(15)

In [ ]:
plt.figure(figsize=(8, 5))
top_coef = bank_coef.head(10).sort_values("coefficient")
plt.barh(top_coef["feature"], top_coef["coefficient"])
plt.title("Bank marketing: strongest logistic regression coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Encoded feature")
plt.show()

## 8. Save model artifacts

The Streamlit app loads these files from the `artifacts/` folder.

In [ ]:
model_path = ARTIFACT_DIR / "bank_logistic_regression_model.joblib"
metadata_path = ARTIFACT_DIR / "bank_logistic_regression_metadata.json"

metrics_at_050 = evaluate_binary_classifier(y_valid, valid_pred, valid_proba).iloc[0].drop("model").to_dict()
metadata = {
    "model_name": "Bank Marketing Logistic Regression",
    "target_column": "y",
    "positive_class_label": "positive response",
    "negative_class_label": "no response",
    "features": X.columns.tolist(),
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "threshold_default": 0.5,
    "random_state": RANDOM_STATE,
    "test_size": 0.25,
    "training_rows": int(len(X_train)),
    "validation_rows": int(len(X_valid)),
    "metrics_at_050": {k: float(v) for k, v in metrics_at_050.items()},
    "source_dataset": "data/bank_marketing_sample.csv",
}

joblib.dump(bank_logreg, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model to: {model_path}")
print(f"Saved metadata to: {metadata_path}")